# Auditoría de disponibilidad y diagnóstico individual

In [1]:
from pathlib import Path
import runpy
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/13_feature_diagnostics.py').is_file())
RECALCULAR = False
if RECALCULAR:
    runpy.run_path(str(ROOT/'src/13_feature_diagnostics.py'), run_name='__main__')
runpy.run_path(str(ROOT/'src/13_render_feature_diagnostics.py'))['render'](False)
display(pd.read_csv(ROOT/'outputs/tables/leakage_feature_availability.csv'))
display(pd.read_csv(ROOT/'outputs/tables/leakage_all_features_scores.csv').groupby('symbol')[['rmse','r2']].agg(['min','mean','max']))


Auditoría individual integrada.


,variable,alias,first_required_opening_offset_hours,last_required_opening_offset_hours,available_by_anchor_plus_hours,prediction_anchor_plus_hours,role,future_target_used_as_input
0,lag_0,close,0,0,1,1,SVR_input,False
1,lag_1,NaN,-1,-1,0,1,SVR_input,False
2,lag_2,NaN,-2,-2,-1,1,SVR_input,False
3,lag_3,NaN,-3,-3,-2,1,SVR_input,False
4,lag_4,NaN,-4,-4,-3,1,SVR_input,False
...,...,...,...,...,...,...,...,...
174,taker_buy_base_asset_volume,NaN,0,0,1,1,diagnostic_only,False
175,taker_buy_quote_asset_volume,NaN,0,0,1,1,diagnostic_only,False
176,return_1h_pct,NaN,-1,0,1,1,diagnostic_only,False
177,abs_return_1h_pct,NaN,-1,0,1,1,diagnostic_only,False


rmse                             r2                    
              min      mean       max        min      mean       max
symbol                                                              
BNBUSDT  0.252537  0.539752  0.936213  -4.771017 -1.836277  0.355154
BTCUSDT  0.215494  0.362317  0.853020  -4.938369 -0.994908  0.338526
ETHUSDT  0.286691  0.523668  0.969551  -8.080618 -2.260278  0.417637
SOLUSDT  0.378507  0.929850  2.126278 -13.144691 -2.884504  0.409517
XRPUSDT  0.404813  0.719675  1.372677  -3.461936 -1.136144  0.394930

### 2.5.3 Diagnóstico individual de todas las entradas

Se reemplaza el diagnóstico limitado a close por **179 regresiones univariadas por activo y fold**: 168 cierres rezagados (`lag_0` es close), las otras ocho columnas numéricas originales, retorno horario, retorno absoluto y volatilidad histórica de 24 horas. Son **4.475 evaluaciones** en los mismos cinco folds cronológicos de la sección 3. Cada OLS con intercepto utiliza una única columna y escalado ajustado únicamente con TRAIN; no hay búsqueda de hiperparámetros. El modelo base sigue siendo el SVR con 168 cierres: estas OLS son diagnósticas.

Se exige la misma elegibilidad y el mismo confinamiento temporal del modelo: historia de 168 cierres, objetivo y referencia completos dentro de cada bloque. Las muestras de entrenamiento y validación coinciden con la auditoría `base_folds.csv`. La referencia constante usa la media de y de TRAIN. RMSE se expresa en puntos porcentuales y R² compara con la media observada del bloque validado. AUC no aplica a regresión.

| Activo | Evaluaciones de rezagos | RMSE mínimo–máximo | R² mínimo–máximo | Alertas R² ≥ 0,8 |
| --- | --- | --- | --- | --- |
| BNBUSDT | 840 | 0.3386 a 0.9362 | -4.7710 a -0.3646 | 0 |
| BTCUSDT | 840 | 0.3272 a 0.3965 | -1.4282 a -0.0679 | 0 |
| ETHUSDT | 840 | 0.3451 a 0.9696 | -8.0806 a -0.0062 | 0 |
| SOLUSDT | 840 | 0.4571 a 2.0484 | -12.1281 a -0.1905 | 0 |
| XRPUSDT | 840 | 0.4925 a 1.0796 | -3.4348 a -0.3066 | 0 |

Los rangos reúnen los 168 rezagos y cinco folds de cada activo, no son intervalos de confianza ni resultados de una selección. Los candidatos originales y derivados se resumen sobre los 25 bloques activo-fold:

| Variable | RMSE medio | R² mínimo–máximo | Alertas R² ≥ 0,8 |
| --- | --- | --- | --- |
| abs_return_1h_pct | 0.4544 | -2.5252 a 0.1615 | 0 |
| high | 0.6193 | -10.7323 a -0.0069 | 0 |
| low | 0.6130 | -9.6104 a -0.0053 | 0 |
| number_of_trades | 0.6234 | -13.1447 a 0.1248 | 0 |
| open | 0.6167 | -10.2647 a -0.0063 | 0 |
| past_volatility_24h_pct | 0.3622 | -0.4974 a 0.4176 | 0 |
| quote_asset_volume | 0.5197 | -10.3582 a 0.1231 | 0 |
| return_1h_pct | 0.5036 | -3.9231 a -0.0038 | 0 |
| taker_buy_base_asset_volume | 0.4567 | -4.0814 a 0.2243 | 0 |
| taker_buy_quote_asset_volume | 0.5196 | -10.1956 a 0.1197 | 0 |
| volume | 0.4559 | -4.0797 a 0.2279 | 0 |

Se registran **0 alertas** con el umbral descriptivo R² ≥ 0,8 de la guía. Un resultado alto es una señal para revisar disponibilidad y alineación, no prueba de fuga; un resultado bajo tampoco acredita su ausencia. Se conservan todas las evaluaciones, sin escoger columnas según su mejor fold ni interpretar el máximo entre miles de evaluaciones como evidencia confirmatoria.

[Resultados de las 4.475 evaluaciones](../outputs/tables/leakage_all_features_scores.csv). El archivo guarda parámetros del escalado, coeficiente, intercepto, conteos y error de la media de TRAIN. No se realizan pruebas de significación ni se usan estos resultados para modificar el modelo.

**Disponibilidad.** Para cada rezago k, el cierre procede de la vela s−k y está disponible nominalmente en s−k+1 hora. Los ocho campos numéricos de la vela actual se consideran disponibles solo cuando termina esa vela, en s+1 hora. El retorno necesita los cierres s−1 y s; la volatilidad pasada necesita los cierres s−24 a s. Todos preceden o coinciden con la emisión nominal s+1 hora. Las claves symbol y fechas no son predictores. La etiqueta futura nunca aparece en las entradas.

[Registro de disponibilidad de las 179 variables](../outputs/tables/leakage_feature_availability.csv). La disponibilidad es nominal tras el cierre; los archivos históricos no registran latencia real de recepción. Un sistema operativo deberá esperar confirmación de la vela cerrada. El objetivo conserva sus propias 24 horas futuras y solo se usa como etiqueta en el ajuste o la evaluación.

La volatilidad pasada comparte fórmula con el objetivo, pero utiliza otro intervalo temporal: no es fuga por ese solo hecho. Se mantienen separadas las variables disponibles, los candidatos excluidos por diseño y los datos futuros prohibidos. Este diagnóstico no certifica ausencia universal de fuga ni decide transformaciones a partir de TEST.